# 01 — Metric harness + base-rate invariance check
Work unit 1 (dataset-independent). Confirms the corrected metric math and the load-bearing assumption: S / FNR / HCFN / ECE_atk are base-rate-invariant.

## Session bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first, every session) ===
# A fresh Colab runtime forgets --global git config, so we re-set it here.
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')          # so `import metrics` finds src/metrics.py
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready - identity set, src/ on path, repo pulled.


## Metric panel on synthetic data

In [2]:
import importlib, metrics
importlib.reload(metrics)
from metrics import ece_attack_conditional
import numpy as np
y = np.r_[np.ones(100), np.zeros(1000)]; p = np.r_[np.full(100,0.004), np.full(1000,0.002)]
print("sanity (expect ~0.996):", round(ece_attack_conditional(y, p, t=0.0213), 4))

sanity (expect ~0.996): 0.996


In [3]:
!grep -n "invariance_check\|prevalence\|savefig\|linspace\|subsample" notebooks/01_metric_harness.ipynb

1:{"cells":[{"cell_type":"markdown","id":"aa572708","metadata":{"id":"aa572708"},"source":["# 01 — Metric harness + base-rate invariance check\n","Work unit 1 (dataset-independent). Confirms the corrected metric math and the load-bearing assumption: S / FNR / HCFN / ECE_atk are base-rate-invariant."]},{"cell_type":"markdown","id":"5e5e2a15","metadata":{"id":"5e5e2a15"},"source":["## Session bootstrap"]},{"cell_type":"code","execution_count":1,"id":"62e0ac80","metadata":{"colab":{"base_uri":"https://localhost:8080/"},"id":"62e0ac80","executionInfo":{"status":"ok","timestamp":1782059054858,"user_tz":-60,"elapsed":40047,"user":{"displayName":"Md Anas Biswas","userId":"02392062433188038282"}},"outputId":"21de7204-6ad9-4143-dd8e-ec903ebbc7e9"},"outputs":[{"output_type":"stream","name":"stdout","text":["Mounted at /content/drive\n","/content/drive/MyDrive/PICALIB_Research/picalib-research\n","Already up to date.\n","Session ready - identity set, src/ on path, repo pulled.\n"]}],"source":["# 

In [4]:
import importlib, metrics; importlib.reload(metrics)
from metrics import (all_metrics, severity_risk_sweep, bootstrap_ci,
    severity_S, base_rate_invariance_check, _synthetic)

y, p = _synthetic(seed=0)
for k, v in all_metrics(y, p, t=0.5).items():
    print(f'{k:14s}: {v}')

FNR           : 0.154
FNCR          : 0.11107180708373865
S             : 0.7212455005437575
HCFN          : 0.05
HCFN_share    : 0.3246753246753247
ECE_atk       : 0.32256549696057785
ECE_pooled    : 0.17640392460033083
deployed_risk : 0.055535903541869325
n_attacks     : 2000
n_misses      : 308


## Bootstrap 95% CI for S

In [5]:
pt, lo, hi = bootstrap_ci(severity_S, y, p, n_boot=1000, t=0.5)
print(f'S = {pt:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')

S = 0.7212  95% CI [0.7012, 0.7413]


## Load-bearing check: invariance under prevalence sweep

In [6]:
rows, passed = base_rate_invariance_check(y, p, t=0.5)
print(f"{'pi':>8}{'S':>9}{'FNR':>9}{'HCFN_sh':>9}{'ECE_atk':>10}{'ECE_pool':>10}")
for r in rows:
    print(f"{r['pi']:>8.3f}{r['S']:>9.4f}{r['FNR']:>9.4f}"
          f"{r['HCFN_share']:>9.4f}{r['ECE_atk']:>10.4f}{r['ECE_pooled']:>10.4f}")
print('\nINVARIANCE CHECK:', 'PASS' if passed else 'FAIL')
assert passed, 'Invariance broken - investigate before building further.'

      pi        S      FNR  HCFN_sh   ECE_atk  ECE_pool
   0.500   0.7212   0.1540   0.3247    0.3226    0.1787
   0.100   0.7212   0.1540   0.3247    0.3226    0.1669
   0.010   0.7212   0.1540   0.3247    0.3226    0.1683
   0.001   0.7212   0.1540   0.3247    0.3226    0.1689

INVARIANCE CHECK: PASS


## Figure: invariants flat, pooled ECE moves

In [ ]:
import matplotlib.pyplot as plt
pis = [r['pi'] for r in rows]
fig, ax = plt.subplots(figsize=(6,4))
for key, lab in [('S','S (severity)'),('FNR','FNR'),
                 ('ECE_atk','ECE_atk'),('ECE_pooled','ECE pooled')]:
    ax.plot(pis, [r[key] for r in rows], marker='o', label=lab)
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel('attack prevalence \u03c0 (log)'); ax.set_ylabel('metric value')
ax.set_title('Base-rate invariance: S/FNR/ECE_atk flat, pooled ECE moves')
ax.legend(); plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/invariance_check.png', dpi=150); plt.show()

---
## Commit + push

In [ ]:
!git add -A
!git commit -m "Metric harness + base-rate invariance figure (folder layout)"
!git push
print('Pushed. Confirm GitHub + Drive in sync.')

[main c671166] Metric harness + base-rate invariance figure (folder layout)
 2 files changed, 290 insertions(+)
 create mode 100644 src/data_loaders.py
 create mode 100644 src/dedup.py
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.93 KiB | 388.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/anasbiswas1/picalib-research.git
   cd60fe4..c671166  main -> main
Pushed. Confirm GitHub + Drive in sync.


In [ ]:
%cd {REPO_DIR}
import os
print("figures/:", os.listdir('figures') if os.path.exists('figures') else "MISSING")
print("reports/:", os.listdir('reports') if os.path.exists('reports') else "not created yet")

/content/drive/MyDrive/PICALIB_Research/picalib-research
figures/: ['invariance_check.png']
reports/: not created yet
